# A6: First MP1 Visualization — Seattle Crime Data

For this assignment, I am creating the first visualizations for Mini Project 1 using the Seattle Police Department Crime Data 2008–Present dataset. These charts will become part of the Analysis section of my MP1b notebook.

Dataset source: https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5/about_data

The CSV file is not included in GitHub because it is larger than GitHub's file size limit. This notebook expects the CSV file to be stored locally in the same folder as this notebook.

## Analytical Questions

1. How do reported crime counts vary by hour of day, day of week, and month in Seattle?
2. Which Seattle precincts, sectors, or neighborhoods have the highest number of reported offenses?
3. How do the most common offense sub-categories differ across time periods, such as morning, afternoon, evening, and night?

## Visualization Plan

This notebook creates three Plotly charts that begin answering the analytical questions above. Each chart is saved as a static image file in the `charts/` folder so it can be committed to GitHub for A6.

In [41]:
import os
import pandas as pd
import plotly.express as px

# Save all A6 chart images into one folder.
os.makedirs("charts", exist_ok=True)

## Load the Dataset

This step loads the Seattle crime CSV into pandas so I can inspect the dataset and prepare it for the Plotly visualizations used in A6.

In [29]:
# Load the SPD crime dataset. The CSV is stored locally because the file is too large for GitHub.
df = pd.read_csv("./SPD_Crime_Data__2008-Present_20260501.csv")

df.head()

,Report Number,Report DateTime,Offense ID,Offense Date,NIBRS Group AB,NIBRS Crime Against Category,Offense Sub Category,Shooting Type Group,Block Address,Latitude,Longitude,Beat,Precinct,Sector,Neighborhood,Reporting Area,Offense Category,NIBRS Offense Code Description,NIBRS_offense_code,Census Block 2020
0,2016-091229,2016 Mar 15 02:56:00 PM,7647511654,2016 Mar 15 02:21:00 PM,B,ANY,ALL OTHER,-,93XX BLOCK OF AURORA AVE N,47.696722,-122.344595,N3,North,N,-,704,ALL OTHER,All Other Offenses,90Z,-
1,2020-152553,2020 May 08 07:26:09 PM,13117279109,2020 May 08 11:40:00 AM,A,PROPERTY,BURGLARY,-,58XX BLOCK OF 5TH AVE NE,47.67117473,-122.32282696274,B3,North,B,WALLINGFORD,1545,PROPERTY CRIME,Burglary/Breaking & Entering,220,4500.2007
2,2026-903462,2026 Feb 22 09:01:19 AM,68657554298,2026 Feb 19 01:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,92XX BLOCK OF 35TH AVE SW,47.52017448,-122.376792267112,F2,Southwest,F,ROXHILL/WESTWOOD/ARBOR HEIGHTS,4930,PROPERTY CRIME,All Other Larceny,23H,11402.3008
3,2015-361048,2015 Oct 15 02:05:00 PM,7693699654,2015 Oct 15 02:05:00 PM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,22XX BLOCK OF E MADISON ST,47.61879546,-122.303098812171,C2,East,C,-,5459,ALL OTHER,False Pretenses/Swindle/Confidence Game,26A,-
4,2016-437709,2016 Dec 05 07:19:00 PM,7695857797,2016 Dec 02 10:00:00 AM,A,PROPERTY,EXTORTION/FRAUD/FORGERY/BRIBERY (INCLUDES BAD ...,-,23XX BLOCK OF FRANKLIN AVE E,47.64087835,-122.324662544481,D3,West,D,-,5107,ALL OTHER,Credit Card/Automated Teller Machine Fraud,26B,-


## Data Profile

Before making the charts, I check the dataset structure and key columns to make sure the time, location, and offense category fields are usable.

In [30]:
# Start by checking the size of the dataset, column types, and missing values.
# This helps me see which fields may need cleaning before visualization.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531135 entries, 0 to 1531134
Data columns (total 20 columns):
 #   Column                          Non-Null Count    Dtype 
---  ------                          --------------    ----- 
 0   Report Number                   1531135 non-null  object
 1   Report DateTime                 1531135 non-null  object
 2   Offense ID                      1531135 non-null  int64 
 3   Offense Date                    1531135 non-null  object
 4   NIBRS Group AB                  1531135 non-null  object
 5   NIBRS Crime Against Category    1531135 non-null  object
 6   Offense Sub Category            1531135 non-null  object
 7   Shooting Type Group             1531135 non-null  object
 8   Block Address                   1531135 non-null  object
 9   Latitude                        1531135 non-null  object
 10  Longitude                       1531135 non-null  object
 11  Beat                            1531135 non-null  object
 12  Precinct      

In [31]:
# Check the exact column names so I can use the right fields later.
df.columns

Index(['Report Number', 'Report DateTime', 'Offense ID', 'Offense Date',
       'NIBRS Group AB', 'NIBRS Crime Against Category',
       'Offense Sub Category', 'Shooting Type Group', 'Block Address',
       'Latitude', 'Longitude', 'Beat', 'Precinct', 'Sector', 'Neighborhood',
       'Reporting Area', 'Offense Category', 'NIBRS Offense Code Description',
       'NIBRS_offense_code', 'Census Block 2020'],
      dtype='object')

In [33]:
# Preview the main fields for my MP1 questions: time, offense category, and location.
df[[
    "Offense Date",
    "Report DateTime",
    "Offense Category",
    "Neighborhood",
    "Precinct",
    "Sector",
    "Beat"
]].head()

,Offense Date,Report DateTime,Offense Category,Neighborhood,Precinct,Sector,Beat
0,2016 Mar 15 02:21:00 PM,2016 Mar 15 02:56:00 PM,ALL OTHER,-,North,N,N3
1,2020 May 08 11:40:00 AM,2020 May 08 07:26:09 PM,PROPERTY CRIME,WALLINGFORD,North,B,B3
2,2026 Feb 19 01:00:00 PM,2026 Feb 22 09:01:19 AM,PROPERTY CRIME,ROXHILL/WESTWOOD/ARBOR HEIGHTS,Southwest,F,F2
3,2015 Oct 15 02:05:00 PM,2015 Oct 15 02:05:00 PM,ALL OTHER,-,East,C,C2
4,2016 Dec 02 10:00:00 AM,2016 Dec 05 07:19:00 PM,ALL OTHER,-,West,D,D3


In [34]:
# Look at the most common offense categories before comparing them across time periods.
df["Offense Category"].value_counts().head(10)

Offense Category
ALL OTHER         750395
PROPERTY CRIME    700037
VIOLENT CRIME      80703
Name: count, dtype: int64

## Data Cleaning and Preparation

I convert the date fields into datetime values and create new time columns for the visual analysis.

In [35]:
# The offense date loaded as text, so I convert it to datetime before extracting hour, day, and month.
df["Offense Date"] = pd.to_datetime(
    df["Offense Date"],
    format="mixed",
    errors="coerce"
)

df["Report DateTime"] = pd.to_datetime(
    df["Report DateTime"],
    format="mixed",
    errors="coerce"
)

# Rows without an offense date cannot be used for my time-based charts.
df = df.dropna(subset=["Offense Date"])

# Create time columns for the first analytical question.
df["Hour"] = df["Offense Date"].dt.hour
df["Month"] = df["Offense Date"].dt.month
df["Year"] = df["Offense Date"].dt.year
df["Day of Week"] = df["Offense Date"].dt.day_name()

df[["Offense Date", "Hour", "Day of Week", "Month", "Year"]].head()

,Offense Date,Hour,Day of Week,Month,Year
0,2016-03-15 14:21:00,14,Tuesday,3,2016
1,2020-05-08 11:40:00,11,Friday,5,2020
2,2026-02-19 13:00:00,13,Thursday,2,2026
3,2015-10-15 14:05:00,14,Thursday,10,2015
4,2016-12-02 10:00:00,10,Friday,12,2016


## Analysis

The following charts answer the three MP1 analytical questions using Plotly. Each chart is saved as a PNG file in the `charts/` folder.

### Chart 1: Reported Crime Counts by Hour of Day

This chart answers Question 1 by showing how reported offense counts vary across the 24 hours of the day.

In [40]:
# To answer my first question, I start with the hour of day.
# I group the records by the "Hour" column and count how many reported offenses fall into each hour.
# This will show whether reported crimes are evenly distributed across the day or concentrated around certain times.
hourly_crimes = (
    df["Hour"]
    .value_counts()
    .sort_index()
    .reset_index()
)

# Rename the columns so the chart code and axis labels are easier to understand.
hourly_crimes.columns = ["hour", "reported_offenses"]

# A bar chart works here because I am comparing counts across ordered hour categories from 0 to 23.
fig1 = px.bar(
    hourly_crimes,
    x="hour",
    y="reported_offenses",
    title="Reported Crime Counts by Hour of Day in Seattle",
    labels={
        "hour": "Hour of Day",
        "reported_offenses": "Number of Reported Offenses"
    }
)

# Save the chart as a PNG file because A6 asks for the charts to be committed as static image files.
fig1.write_image("charts/crime_counts_by_hour.png")

### Chart 2: Top 10 Seattle Neighborhoods by Reported Offenses

This chart answers Question 2 by comparing which Seattle neighborhoods have the highest number of reported offenses. I use a horizontal bar chart because the neighborhood names are easier to read this way.

In [46]:
# To answer my second question, I count reported offenses by neighborhood.
# I remove "-" because it does not represent an actual neighborhood; it is more like missing or unassigned location data.
# Keeping only the top 10 actual neighborhoods makes the chart more meaningful.
neighborhood_counts = (
    df[df["Neighborhood"] != "-"]["Neighborhood"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)

# Rename the columns so the chart labels are clearer.
neighborhood_counts.columns = ["neighborhood", "reported_offenses"]

# Reverse the order so the largest bar appears at the top in the horizontal chart.
neighborhood_counts = neighborhood_counts.sort_values("reported_offenses", ascending=True)

# A horizontal bar chart works better here because neighborhood names can be long.
# It compares reported offense counts across neighborhood categories.
fig2 = px.bar(
    neighborhood_counts,
    x="reported_offenses",
    y="neighborhood",
    orientation="h",
    title="Top 10 Seattle Neighborhoods by Reported Offenses",
    labels={
        "reported_offenses": "Number of Reported Offenses",
        "neighborhood": "Neighborhood"
    }
)

# Save the chart as a PNG file for the A6 submission.
fig2.write_image("charts/top_10_neighborhoods_by_reported_offenses.png")

### Chart 3: Most Common Offense Sub-Categories by Time Period

This chart answers Question 3 by grouping offense times into morning, afternoon, evening, and night, then comparing the most common specific offense sub-categories across those periods.

In [51]:
# To answer my third question, I use offense sub-categories instead of the broad "Offense Category" field.
# The broad field only has three groups, while sub-categories show more specific offense types.
def assign_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["Time Period"] = df["Hour"].apply(assign_time_period)

# Remove vague categories like "ALL OTHER" and "999" so the chart focuses on meaningful offense types.
filtered_subcategories = df[
    ~df["Offense Sub Category"].isin(["ALL OTHER", "999"])
]

# I keep the four most common offense sub-categories so the grouped chart stays readable.
# This focuses the comparison on the categories that appear most often in the dataset.
top_sub_categories = (
    filtered_subcategories["Offense Sub Category"]
    .dropna()
    .value_counts()
    .head(4)
    .index
)

# Count reports for each combination of time period and offense sub-category.
# This lets me compare whether common offense types appear more often in certain parts of the day.
subcategory_time_counts = (
    filtered_subcategories[
        filtered_subcategories["Offense Sub Category"].isin(top_sub_categories)
    ]
    .groupby(["Time Period", "Offense Sub Category"])
    .size()
    .reset_index(name="reported_offenses")
)

# A grouped bar chart works here because I am comparing several offense sub-categories within each time period.
fig3 = px.bar(
    subcategory_time_counts,
    x="Time Period",
    y="reported_offenses",
    color="Offense Sub Category",
    barmode="group",
    title="Most Common Offense Sub-Categories by Time Period in Seattle",
    labels={
        "Time Period": "Time Period",
        "reported_offenses": "Number of Reported Offenses",
        "Offense Sub Category": "Offense Sub-Category"
    },
    category_orders={
        "Time Period": ["Morning", "Afternoon", "Evening", "Night"]
    }
)

fig3.write_image("charts/offense_subcategories_by_time_period.png")

## Conclusions

These three charts begin answering my MP1 analytical questions. The first chart shows how reported offenses vary by hour of day, the second chart shows which neighborhoods have the highest number of reported offenses, and the third chart compares common offense sub-categories across morning, afternoon, evening, and night.

Together, the charts show temporal patterns, spatial concentration, and offense-type differences in Seattle crime reports. I will use these visualizations as the starting point for the Analysis section of my MP1b notebook.